In [ ]:
# Cell 1: Import thư viện và nạp dữ liệu
import sys
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import joblib

sys.path.append(os.path.abspath(".."))
from deep_learning.lstm_model import StockLSTM

# Nạp dữ liệu ML-ready đã chuẩn bị từ Notebook 06
df = pd.read_parquet("../data/processed/AAPL_ml_ready.parquet")
feature_cols = [c for c in df.columns if c not in ['Target', 'Close']]

# Cell 2: Chuẩn hóa dữ liệu (Scaling) & Tạo 3D Sliding Window
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[feature_cols])

# Lưu Scaler lại để dùng khi Live Trading
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/lstm_scaler.pkl")

# Hàm tạo cửa sổ trượt (Time Steps)
def create_sequences(data, targets, seq_length=30):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(targets[i + seq_length])
    return np.array(X), np.array(y)

SEQ_LEN = 30  # Nhìn lại 30 ngày quá khứ
X_seq, y_seq = create_sequences(scaled_features, df['Target'].values, seq_length=SEQ_LEN)

# Chia Train/Test theo thời gian (80% Train, 20% Test)
split = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

# Chuyển đổi sang PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=False)

# Cell 3: Vòng lặp Huấn luyện (Training Loop)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StockLSTM(input_size=len(feature_cols), hidden_size=64, num_layers=2).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
print(f"Bắt đầu huấn luyện LSTM trên thiết bị: {device}")

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
                
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/len(train_loader):.4f}")

# Cell 4: Đánh giá và Lưu Model trọng số (.pth)
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t.to(device)).cpu().numpy()
    acc = ((test_preds > 0.5) == y_test_t.numpy()).mean()
    print(f"\nAccuracy trên tập Test của LSTM: {acc * 100:.2f}%")

# Lưu trọng số vào thư mục models/
torch.save(model.state_dict(), "../models/lstm_aapl.pth")
print("✅ Đã lưu trọng số mô hình PyTorch tại: ../models/lstm_aapl.pth")


ModuleNotFoundError: No module named 'torch'